# Активное обучение для распознавания дорожных знаков GTSRB

Блокнот показывает главное из проекта: подготовку данных, модель `TrafficSignCNN`, критерии неопределенности и полный цикл pool-based active learning. Основная реализация находится в `active_learning_gtsrb.py`, поэтому здесь нет дублирования кода.

## Идея метода

На каждом раунде модель обучается на размеченном множестве $L$, оценивает изображения из пула $U$ и запрашивает метки только для наиболее информативного пакета $Q$.

Используемые оценки неопределенности:

$$a_{\mathrm{LC}}(x)=1-\max_c p_\theta(y=c\mid x),$$

$$a_{\mathrm{M}}(x)=-\left(p_{(1)}(x)-p_{(2)}(x)\right),$$

$$a_{\mathrm{H}}(x)=-\sum_{c=1}^{C}p_\theta(y=c\mid x)\log p_\theta(y=c\mid x).$$

In [ ]:
from dataclasses import asdict
from pathlib import Path
import csv
import json

import matplotlib.pyplot as plt
import numpy as np

from active_learning_gtsrb import (
    RunConfig,
    choose_device,
    label_of,
    make_datasets,
    plot_learning_curves,
    run_strategy,
    seed_everything,
    uncertainty_scores,
    write_csv,
)

## Компактная конфигурация

Параметры ниже подходят для учебного запуска. Для полного эксперимента увеличьте `rounds`, `epochs` и снимите ограничения `max_train_samples` и `max_test_samples`.

In [ ]:
config = RunConfig(
    data_dir="data",
    output_dir="results_notebook",
    strategies=("entropy", "margin", "random"),
    rounds=2,
    query_size=200,
    initial_per_class=5,
    epochs=2,
    batch_size=128,
    learning_rate=1e-3,
    weight_decay=1e-4,
    validation_fraction=0.1,
    seed=42,
    num_workers=0,
    image_size=48,
    max_train_samples=5000,
    max_test_samples=2000,
    smoke_test=False,
)

device = choose_device()
print(f"Устройство: {device}")
print(json.dumps(asdict(config), ensure_ascii=False, indent=2))

## Как стратегии оценивают неопределенность

Ниже используются три искусственных распределения вероятностей. Равномерное распределение должно получить наибольшую энтропию и наименьший разрыв между двумя лучшими классами.

In [ ]:
probabilities = np.array([
    [0.90, 0.05, 0.05],
    [0.48, 0.47, 0.05],
    [1 / 3, 1 / 3, 1 / 3],
])
names = ["уверенное", "два близких класса", "равномерное"]
strategies = ["entropy", "margin", "least_confidence"]
scores = {name: uncertainty_scores(probabilities, name) for name in strategies}

for row_id, distribution_name in enumerate(names):
    values = ", ".join(f"{strategy}: {scores[strategy][row_id]:.3f}" for strategy in strategies)
    print(f"{distribution_name:20s} | {values}")

In [ ]:
x = np.arange(len(names))
width = 0.24
fig, ax = plt.subplots(figsize=(9, 4.5))
for offset, strategy in enumerate(strategies):
    ax.bar(x + (offset - 1) * width, scores[strategy], width, label=strategy)
ax.set_xticks(x, names)
ax.set_ylabel("Оценка информативности")
ax.set_title("Сравнение критериев неопределенности")
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## Просмотр GTSRB

Следующая ячейка загружает GTSRB и показывает примеры. Установите `LOAD_PREVIEW = True`, когда будете готовы к загрузке датасета.

In [ ]:
LOAD_PREVIEW = False

if LOAD_PREVIEW:
    train_dataset, pool_dataset, test_dataset, train_indices, test_indices = make_datasets(config)
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for ax, index in zip(axes.flat, train_indices[:10]):
        image, class_id = pool_dataset[index]
        image = image.permute(1, 2, 0).numpy()
        mean = np.array([0.3337, 0.3064, 0.3171])
        std = np.array([0.2672, 0.2564, 0.2629])
        ax.imshow(np.clip(image * std + mean, 0, 1))
        ax.set_title(f"Класс {class_id}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Предпросмотр отключен. Установите LOAD_PREVIEW = True.")

## Запуск активного обучения

Ячейка выполняет один и тот же эксперимент для `entropy`, `margin` и `random`. Она выключена по умолчанию, поскольку даже компактный запуск обучает девять моделей. Для старта установите `RUN_EXPERIMENT = True`.

In [ ]:
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    seed_everything(config.seed)
    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "run_config.json").write_text(
        json.dumps(asdict(config), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    train_dataset, pool_dataset, test_dataset, train_indices, test_indices = make_datasets(config)
    all_metrics = []
    all_selected = []
    for strategy in config.strategies:
        metrics, selected = run_strategy(
            strategy, train_dataset, pool_dataset, test_dataset,
            train_indices, test_indices, config, device
        )
        all_metrics.extend(metrics)
        all_selected.extend(selected)
    write_csv(output_dir / "metrics.csv", all_metrics)
    write_csv(output_dir / "selected_samples.csv", all_selected)
    plot_learning_curves(all_metrics, output_dir / "learning_curve.png")
    print(f"Результаты сохранены в {output_dir.resolve()}")
else:
    print("Эксперимент отключен. Установите RUN_EXPERIMENT = True.")

## Анализ результатов

После обучения ячейка читает `metrics.csv` и строит обе основные метрики. Чем выше кривая при одинаковом числе размеченных изображений, тем эффективнее стратегия использует бюджет разметки.

In [ ]:
metrics_path = Path(config.output_dir) / "metrics.csv"
if metrics_path.exists():
    with metrics_path.open(encoding="utf-8") as file:
        rows = list(csv.DictReader(file))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for strategy in config.strategies:
        selected = [row for row in rows if row["strategy"] == strategy]
        labeled = [int(row["labeled_count"]) for row in selected]
        axes[0].plot(labeled, [float(row["test_accuracy"]) for row in selected], "o-", label=strategy)
        axes[1].plot(labeled, [float(row["test_macro_f1"]) for row in selected], "o-", label=strategy)
    axes[0].set_title("Test accuracy")
    axes[1].set_title("Test macro-F1")
    for ax in axes:
        ax.set_xlabel("Количество размеченных изображений")
        ax.grid(alpha=0.25)
        ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Файл metrics.csv пока не создан. Сначала запустите эксперимент.")

## Что считать результатом

Активная стратегия полезна, если ее кривые `accuracy` и `macro-F1` устойчиво выше `random` при одинаковом размере $L$. Для вывода в курсовой эксперимент следует повторить с несколькими значениями `seed` и сравнить среднее значение и стандартное отклонение.